In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from copy import deepcopy

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import OrdinalEncoder

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

Libraries loaded.


In [2]:
df01 = pd.read_csv('../data/Interim/01_sales_index.csv')
df04 = pd.read_csv('../data/Interim/04_high_grown_prices.csv')
df05 = pd.read_csv('../data/Interim/05_low_grown_prices.csv')
df06 = pd.read_csv('../data/Interim/06_offgrade_dust_prices.csv')
df09 = pd.read_csv('../data/Interim/09_weather_features.csv')

print(f'01 sales index  : {df01.shape}')
print(f'04 high grown   : {df04.shape}')
print(f'05 low grown    : {df05.shape}')
print(f'06 off/dust     : {df06.shape}')
print(f'09 weather      : {df09.shape}')

01 sales index  : (26, 108)
04 high grown   : (516, 6)
05 low grown    : (1348, 6)
06 off/dust     : (1170, 6)
09 weather      : (100, 61)


In [5]:
# ── Sale-level context: minimum columns needed for baseline ─────────────────
MONTH_ORDER = {
    'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
    'July':7,'August':8,'September':9,'October':10,'November':11,'December':12
}

# Core sale identity + supply + market + weather
# Segment-specific gross revenue columns are added per-segment below.
SALE_COLS_CORE = [
    # Identity
    'sale_id', 'sale_number', 'sale_year', 'sale_month',
    # Supply
    'total_lots', 'total_kgs', 'reprint_lots', 'reprint_quantity',
    # Market mood
    'sentiment_overall', 'sentiment_ex_estate',
    # Volume + macro
    'total_sold_weekly_2026', 'public_auction_weekly_2026',
    'sl_production_mkgs', 'fx_usd_2026',
    # Weather — the primary research driver
    'avg_weather_severity',
    'western_nuwara_eliya_weather_score',
    'uva_udapussellawa_weather_score',
    'low_grown_weather_score',
    'crop_nuwara_eliya_trend', 'crop_uva_trend', 'crop_low_grown_trend',
]

sale_meta = df01[[c for c in SALE_COLS_CORE if c in df01.columns]].copy()

# Derived columns
sale_meta['sale_month_enc']       = sale_meta['sale_month'].map(MONTH_ORDER).fillna(0).astype(int)
sale_meta['is_production_known']  = sale_meta['sl_production_mkgs'].notna().astype(int)
sale_meta['sl_production_mkgs']   = sale_meta['sl_production_mkgs'].fillna(
                                        sale_meta['sl_production_mkgs'].mean())
sale_meta['volume_yoy_change_pct']= (
    (df01['total_sold_weekly_2026'].fillna(0) - df01['total_sold_weekly_2025'].fillna(0))
    / df01['total_sold_weekly_2025'].replace(0, float('nan'))
).fillna(0)

# Sort chronologically and assign rank
sale_meta['sort_key'] = (
    sale_meta['sale_year'].fillna(9999) * 100 + sale_meta['sale_number'].fillna(99))
sale_meta = sale_meta.sort_values('sort_key').reset_index(drop=True)
sale_meta['sale_rank'] = sale_meta.index

print(f'Sales loaded : {len(sale_meta)} | Rank range: '
      f'{sale_meta["sale_rank"].min()} – {sale_meta["sale_rank"].max()}')
print(f'Columns      : {len(sale_meta.columns)}')
print(sale_meta[['sale_id', 'sale_rank', 'sale_year', 'sale_number']].to_string())


Sales loaded : 26 | Rank range: 0 – 25
Columns      : 26
                                 sale_id  sale_rank  sale_year  sale_number
0                           SALE_34_2025          0     2025.0           34
1                           SALE_35_2025          1     2025.0           35
2                           SALE_36_2025          2     2025.0           36
3                           SALE_38_2025          3     2025.0           38
4                           SALE_39_2025          4     2025.0           39
5                           SALE_40_2025          5     2025.0           40
6                           SALE_41_2025          6     2025.0           41
7                           SALE_42_2025          7     2025.0           42
8                           SALE_43_2025          8     2025.0           43
9                           SALE_44_2025          9     2025.0           44
10                          SALE_45_2025         10     2025.0           45
11                          SAL

In [8]:
# ── Pivot weather features wide by region ─────────────────────────────────────
wx = df09.copy()

# Derive composite average precipitation (not in interim file — compute here)
precip_cols = [c for c in wx.columns if 'precipitation_sum_total' in c and 'lag' not in c
               and not c.startswith('all_')]
if precip_cols:
    wx['all_regions__avg_precipitation'] = wx[precip_cols].mean(axis=1)
else:
    wx['all_regions__avg_precipitation'] = 0.0

def wx_cols(region_prefix):
    return [c for c in wx.columns if c.startswith(region_prefix + '__')]

WX_WESTERN  = wx_cols('western_high')
WX_NUWARA   = wx_cols('nuwara_eliya')
WX_UVA      = wx_cols('uva_udapussellawa')
WX_LOWGROWN = wx_cols('low_grown')
WX_ALL      = WX_WESTERN + WX_NUWARA + WX_UVA + WX_LOWGROWN + ['all_regions__avg_precipitation']

# High grown relevant: western + nuwara + uva (no low grown)
WX_HIGH_GROWN = WX_WESTERN + WX_NUWARA + WX_UVA + ['all_regions__avg_precipitation']
# Low grown relevant: low grown region
WX_LOW_GROWN  = WX_LOWGROWN + ['all_regions__avg_precipitation']

print(f'Weather cols — western_high: {len(WX_WESTERN)}, nuwara_eliya: {len(WX_NUWARA)}')
print(f'               uva_uda: {len(WX_UVA)}, low_grown: {len(WX_LOWGROWN)}')
print(f'High Grown weather features: {len(WX_HIGH_GROWN)}')
print(f'Low Grown  weather features: {len(WX_LOW_GROWN)}')
print(f'Off-Grade/Dust weather features: {len(WX_ALL)}')

Weather cols — western_high: 0, nuwara_eliya: 0
               uva_uda: 0, low_grown: 0
High Grown weather features: 1
Low Grown  weather features: 1
Off-Grade/Dust weather features: 1
